# Wahlauswertung

Die Klasse `ElectionEvaluator` führt die drei Auswertungsschritte automatisch in der richtigen Reihenfolge durch:
1. Mandatszuteilung (`SeatDistributor`)
2. Wahlkreisanzahlzuordnung (`ConstituencyCountDeterminer`)
3. Wahlkreiszuordnung (`ConstituencyAssigner`)

Dieses Notebook zeigt den Standardweg über `evaluate()` sowie die einzelnen Schritte mit ihren Konfigurationsoptionen.

Für eine konzeptionelle Übersicht siehe [Einführung](../../docs/source/de/einfuehrung.md).

In [ ]:
import pandas as pd
from ipres import (
    Election, ElectionConfig, ConstituenciesConfig,
    ElectionEvaluator, SeatDistributor, ConstituencyCountDeterminer, ConstituencyAssigner,
    SeatDistributionMethod, SuperMajorityMargin, MarginUnit,
    VoteMatrixAnalyzer,
)
from ipres.election_config import (
    ConstituencyRepresentation, QuotaCorrectionStrategy,
)
from ipres.allocation import ConstituencyAllocationMethod

## Setup

Für alle Beispiele in diesem Notebook wird dieselbe Wahl verwendet: 5 Wahlkreise, 3 Parteien, nicht-uniforme Stimmenverteilung. A ist der Wahlgewinner.

In [ ]:
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['WK1', 'WK2', 'WK3', 'WK4', 'WK5'],
    'constituency_size': [100_000] * 5,
}))
config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    seed=42,
)

votes = {
    'WK1': {'A': 70, 'B': 20, 'C': 10},
    'WK2': {'A': 50, 'B': 35, 'C': 15},
    'WK3': {'A': 55, 'B': 25, 'C': 20},
    'WK4': {'A': 65, 'B': 25, 'C': 10},
    'WK5': {'A': 60, 'B': 30, 'C': 10},
}

election = Election(electionConfig=config)
iteration = election.start(votes=votes)

---

## `ElectionEvaluator.evaluate()` — Der einfache Weg

`evaluate()` führt alle drei Auswertungsschritte in einem einzigen Aufruf durch und gibt ein `ElectionResult`-Objekt zurück.

In [ ]:
evaluator = ElectionEvaluator(seed=42)
result = evaluator.evaluate(election)

In [ ]:
result.get_seat_distribution_table()

In [ ]:
result.get_constituency_summary_table()

In [ ]:
result.get_constituency_assignment_table()

In [ ]:
x = result.plot_seat_share_pie(min_seats_for_display=1)

---

## Schritt 1: Mandatszuteilung — `SeatDistributor`

Der `SeatDistributor` verteilt die Parlamentssitze auf die Parteien. Es gibt zwei Fälle:
- **Fall 1 (zugewiesene Mehrheit)**: Die Gewinnerpartei erhält die Regierungsmehrheit direkt zugeteilt, die restlichen Sitze gehen proportional an die übrigen Parteien.
- **Fall 2 (proportional)**: Alle Parteien erhalten ihre Sitze proportional zum Wahlergebnis.

Details siehe [Mandatszuteilung](../../docs/source/de/mandats_zuteilung.md).

### `seat_distribution_method`

Für die proportionale Sitzverteilung stehen drei Zuteilungsverfahren zur Auswahl: `SAINTE_LAGUE` (Standard), `D_HONDT` und `HARE_NIEMEYER`. In diesem einfachen Beispiel ergibt sich bei allen Verfahren die gleiche Sitzverteilung.

In [ ]:
for method in SeatDistributionMethod:
    sitze = SeatDistributor(seat_distribution_method=method).run(election)
    print(f"{method.name:15s} → {sitze}")

---

## Schritt 2: Wahlkreisanzahlzuordnung — `ConstituencyCountDeterminer`

Die Hälfte der einer Partei zustehenden Parlamentssitze wird über Direktmandate in Wahlkreisen besetzt. Daher entspricht die Anzahl der Wahlkreise einer Partei ungefähr der Hälfte ihrer Parlamentssitze (`sitze // 2`).

Details siehe [Wahlkreisanzahlzuordnung](../../docs/source/de/wahlkreis_anzahl_zuordnung.md).

### `quota_correction_strategy`

Bei Parteien mit **ungerader** Sitzzahl entsteht durch die ganzzahlige Division ein Defizit: Die Summe der Grundzuteilungen ist kleiner als die Gesamtzahl der Wahlkreise. Die Korrekturstrategie entscheidet, welche Partei das fehlende `+1` erhält.

Im Beispiel (5 Wahlkreise): A=6, B=3, C=1 Sitze → Grundzuteilung A=3, B=1, C=0 → Summe=4, Defizit=1. Nur B (3 Sitze) und C (1 Sitz) haben ungerade Sitzzahlen.

In [ ]:
sitze = SeatDistributor().run(election)
print("Sitze:          ", sitze)
print("Grundzuteilung:", {p: s // 2 for p, s in sitze.items()})
print(f"Summe Grundzuteilung: {sum(s // 2 for s in sitze.values())}  (Wahlkreise: 5 → Defizit: 1)")
print()

for strategy in [QuotaCorrectionStrategy.FAVOR_LARGE_PARTIES,
                 QuotaCorrectionStrategy.FAVOR_SMALL_PARTIES]:
    anzahl = ConstituencyCountDeterminer(quota_correction_strategy=strategy).run(election, sitze)
    print(f"{strategy.name}: {anzahl}")

### `constituency_representation`

Dieser Parameter steuert, welche Parteien überhaupt Wahlkreise erhalten:
- **`ENTIRE_PARLIAMENT`** *(Standard)*: Alle Parteien erhalten Wahlkreise proportional zu ihren Sitzen.
- **`GOVERNING_MAJORITY`**: Nur die Regierungspartei(en) erhalten Wahlkreise. Da alle Wahlkreise trotzdem vertreten sein müssen, muss das Parlament größer sein — `ElectionConfig` berechnet die nötige Parlamentsgröße automatisch, wenn dieser Wert dort gesetzt ist.

Der Modus wird **nicht** direkt an `ConstituencyCountDeterminer` oder `ConstituencyAssigner` übergeben, sondern wird von diesen automatisch aus `election.electionConfig.constituency_representation` gelesen. Für den Vergleich werden daher zwei separate `Election`-Objekte mit unterschiedlicher `ElectionConfig` benötigt.

Im Vergleich: `ENTIRE_PARLIAMENT` ergibt ein Parlament mit 10 Sitzen (2 × 5 Wahlkreise), `GOVERNING_MAJORITY` ergibt 18 Sitze, damit die Regierungsmehrheit allein alle 5 Wahlkreise abdecken kann.

In [ ]:
# ENTIRE_PARLIAMENT: bestehende Election (10 Sitze)
anzahl_ep = ConstituencyCountDeterminer().run(election, sitze)
print(f"ENTIRE_PARLIAMENT (10 Sitze):  {anzahl_ep}  → Summe: {sum(anzahl_ep.values())}")

# GOVERNING_MAJORITY: neue Election mit passendem Parlamentskonfiguration (18 Sitze)
config_gm = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    constituency_representation=ConstituencyRepresentation.GOVERNING_MAJORITY,
    seed=42,
)
election_gm = Election(electionConfig=config_gm)
election_gm.start(votes=votes)
sitze_gm = SeatDistributor().run(election_gm)
anzahl_gm = ConstituencyCountDeterminer().run(election_gm, sitze_gm)
print(f"GOVERNING_MAJORITY ({config_gm.parliamentarySeats} Sitze): {anzahl_gm}  → Summe: {sum(anzahl_gm.values())}")

---

## Schritt 3: Wahlkreiszuordnung — `ConstituencyAssigner`

Im letzten Schritt wird jedem Wahlkreis genau eine Partei zur Vertretung zugewiesen. Die Grundidee: Ein Wahlkreis soll von der Partei vertreten werden, für die er relativ zu ihren anderen Wahlkreisen am wichtigsten ist.

Details siehe [Wahlkreiszuordnung](../../docs/source/de/partei_wahlkreis_zuordnung.md).

### Wichtigkeitsmatrix

Die Wichtigkeit `w_ij` eines Wahlkreises `i` für Partei `j` berechnet sich als:

```
w_ij = (M − 1) · r_ij / Σ(r_kj  für alle k ≠ i)
```

- `w > 1.0` → Wahlkreis ist für diese Partei überdurchschnittlich wichtig
- `w < 1.0` → unterdurchschnittlich wichtig
- `w = 1.0` → Durchschnitt

In [ ]:
vm = election.getLastIteration().vote_matrix.getVotes()
VoteMatrixAnalyzer(vm).show_constituency_importance_matrix()

### `constituency_allocation_method`

Drei Methoden stehen zur Auswahl, um aus der Wichtigkeitsmatrix eine Zuteilung abzuleiten:
- **`OPTIMAL`** *(Standard)*: Ungarischer Algorithmus — maximiert den globalen Gesamtwichtigkeitswert.
- **`GREEDY`**: Weist iterativ das Paar mit dem höchsten Wichtigkeitswert zu.
- **`STABLE_MATCHING`**: Gale-Shapley-Algorithmus — keine stabile Blockierungspaare.

In [ ]:
anzahl = ConstituencyCountDeterminer().run(election, sitze)  # A=3, B=2, C=0
for method in ConstituencyAllocationMethod:
    zuordnung = ConstituencyAssigner(constituency_allocation_method=method, seed=42).run(election, anzahl)
    print(f"{method.name:16s} → {zuordnung}")

---

## Zusammenfassung

| Parameter | Klasse | Standard |
|---|---|---|
| `seat_distribution_method` | `ElectionEvaluator` / `SeatDistributor` | `SAINTE_LAGUE` |
| `quota_correction_strategy` | `ElectionEvaluator` / `ConstituencyCountDeterminer` | `FAVOR_LARGE_PARTIES` |
| `constituency_allocation_method` | `ElectionEvaluator` / `ConstituencyAssigner` | `OPTIMAL` |

`constituency_representation` wird nicht an `ConstituencyCountDeterminer`, `ConstituencyAssigner` oder `ElectionEvaluator` übergeben, sondern automatisch aus `election.electionConfig.constituency_representation` gelesen.